#### Problema 2 - Enriquecimiento y Clasificación de CVEs


Objetivo:
crapee  los  CVEs  publicados  en  los  últimos  3  meses  desde  cvedetails.com.  Buscar los campos  CVE  ID,  fecha  de  publicación,  CVSS  score,  tipo  de 
vulnerabilidad, vendor y producto afectado.


In [ ]:
#!pip install selenium

In [ ]:
# imports
import pandas as pd
import numpy as np
import json
import re
import time
import os

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

from datetime import datetime, timedelta


from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import google.generativeai as genai

#configuración

# API
genai.configure(api_key='') #decido anonimizar mi api key 

MODEL_NAME = "gemini-2.5-flash" #el gemini 3 tarda bastante mas en responder
model = genai.GenerativeModel(MODEL_NAME)

# Archivos
INPUT_FILE = "CVE_3_meses.csv"
OUTPUT_FILE = "cve_final.csv"
CACHE_FILE = "semantic_cache.json"

# Parámetros
BATCH_SIZE = 40
THRESHOLD = 0.9

In [ ]:
# scraping a 3 meses
# 
def crear_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage") # CRÍTICO para no saturar la RAM
    options.add_argument("--memory-pressure-off")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    return webdriver.Chrome(options=options)

def scraping_industrial_3_meses(pagina_inicio=1):
    # --- CÁLCULO DINÁMICO DE FECHA ---
    hoy = datetime.now()
    fecha_90_dias_atras = hoy - timedelta(days=90)
    fecha_filtro = fecha_90_dias_atras.strftime("%Y-%m-%d")
    print(f"Iniciando scraping desde: {fecha_filtro} (90 días atrás desde hoy {hoy.strftime('%Y-%m-%d')})")
    archivo_csv = "CVE_3_meses.csv"
    page_num = pagina_inicio
    
    driver = crear_driver()
    
    try:
        while True:
            # Reinicio de motor cada 15 páginas para evitar el error de sesión
            if page_num % 15 == 0:
                print(f"--- Limpiando memoria. Reiniciando navegador en página {page_num} ---")
                driver.quit()
                driver = crear_driver()

            url = f"https://www.cvedetails.com/vulnerability-search.php?f=1&publishdatestart={fecha_filtro}&page={page_num}"
            print(f"Extrayendo página {page_num} de ~750...")
            
            try:
                driver.get(url)
                time.sleep(4) # Tiempo para el render de Vue.js
                
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                cve_blocks = soup.find_all("div", {"data-tsvfield": "cveinfo"})
                
                if not cve_blocks:
                    print("¡Fin del dataset alcanzado!")
                    break
                
                pagina_data = []
                for block in cve_blocks:
                    try:
                        pagina_data.append({
                            "CVE_ID": block.find(attrs={"data-tsvfield": "cveId"}).get_text(strip=True),
                            "Published": block.find(attrs={"data-tsvfield": "publishDate"}).get_text(strip=True),
                            "CVSS": block.find(attrs={"data-tsvfield": "maxCvssBaseScore"}).get_text(strip=True),
                            "Summary": block.find(attrs={"data-tsvfield": "summary"}).get_text(strip=True)
                        })
                    except:
                        continue
                
                # GUARDADO INMEDIATO (Modo Append)
                df_temp = pd.DataFrame(pagina_data)
                if not os.path.isfile(archivo_csv):
                    df_temp.to_csv(archivo_csv, index=False)
                else:
                    df_temp.to_csv(archivo_csv, mode='a', header=False, index=False)
                
                page_num += 1
                
            except Exception as e:
                print(f"Falla en página {page_num}: {e}. Reintentando con nuevo driver...")
                driver.quit()
                time.sleep(5)
                driver = crear_driver()
                continue

    finally:
        driver.quit()
        print(f"Proceso finalizado. Revisa el archivo {archivo_csv}")

# Ejecutar desde la página donde te quedaste (ej: 1)
scraping_industrial_3_meses(pagina_inicio=1)

df_scraped = pd.read_csv("CVE_3_meses.csv")

df_scraped.drop_duplicates(inplace=True)
df_scraped.reset_index(drop=True, inplace=True)


# esto se demora aprox 1 hora.

# devuelve un dataframe con las columnas CVE_ID, Published, CVSS, Summary

# la estrategia es de Summmary extraer vendor, producto y tipo de vulnerabilidad.

In [ ]:
# Modelo de embeddings

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# prototipos semanticos, para hacer la clasificación rapida con el embedding

prototipos = {
    "Sql Injection": "sql injection database exploit",
    "XSS": "cross site scripting javascript",
    "Overflow": "buffer overflow memory crash",
    "Memory Corruption": "use after free segmentation fault",
    "CSRF": "cross site request forgery",
    "XXE": "xml external entity",
    "SSRF": "server side request forgery",
    "Open Redirect": "url redirect phishing",
    "File Inclusion": "lfi rfi include file",
    "Directory Traversal": "path traversal ../",
    "Code Execution": "remote code execution rce",
    "Privilege Escalation": "gain root privileges",
    "Bypass": "authentication bypass",
    "Denial of Service": "dos crash exhaustion",
    "Information Leak": "data leak sensitive information",
    "Input Validation": "invalid input unsanitized"
}

proto_labels = list(prototipos.keys())
proto_embeddings = embed_model.encode(list(prototipos.values()))

# función para clasificar por similitud semantica

def clasificar_batch(textos):
    emb = embed_model.encode(textos, batch_size=64)
    sims = cosine_similarity(emb, proto_embeddings)
    idx = np.argmax(sims, axis=1)
    return [proto_labels[i] for i in idx]

# cache semántico

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache_data = json.load(f)
else:
    cache_data = []

def es_valido(res):
    invalidos = ["", "n/a", "na", "unknown", "error"]
    return (
        res.get("v","").lower() not in invalidos and
        res.get("p","").lower() not in invalidos
    )

# busqueda en cache

def buscar_cache(textos):
    if not cache_data:
        return [None]*len(textos), embed_model.encode(textos)

    cache_embs = np.array([c["embedding"] for c in cache_data])
    emb = embed_model.encode(textos)

    sims = cosine_similarity(emb, cache_embs)

    resultados = []
    for i in range(len(textos)):
        idx = np.argmax(sims[i])
        if sims[i][idx] >= THRESHOLD:
            resultados.append(cache_data[idx]["result"])
        else:
            resultados.append(None)

    return resultados, emb

# llamada a IA para clasificar vendor y producto

def call_gemini(prompt):
    for _ in range(3):
        try:
            r = model.generate_content(prompt)

            if hasattr(r, "text") and r.text:
                txt = r.text.strip()
            else:
                txt = r.candidates[0].content.parts[0].text.strip()

            txt = txt.replace("```json","").replace("```","").strip()

            match = re.search(r'\[.*\]', txt, re.DOTALL)
            if match:
                return json.loads(match.group(0))

        except Exception:
            time.sleep(2)

    return None

# procesamiento en batch pera evitar timeouts y optimizar llamadas a la IA

def procesar_batch(lista):
    resultados, embeddings = buscar_cache(lista)

    idx_ia = [i for i, r in enumerate(resultados) if r is None]

    if idx_ia:
        textos_ia = [lista[i] for i in idx_ia]

        prompt = f"""
Extract vendor and product.

Return ONLY JSON:
[{{"v":"","p":""}}]

Texts:
{textos_ia}
"""

        data = call_gemini(prompt)

        if not data:
            data = [{"v":"N/A","p":"N/A"}]*len(textos_ia)

        for i, idx in enumerate(idx_ia):
            resultados[idx] = data[i]

            if es_valido(data[i]):
                cache_data.append({
                    "text": lista[idx],
                    "embedding": embeddings[idx].tolist(),
                    "result": data[i]
                })

    return resultados



In [ ]:
# pipeline principal

df = pd.read_csv(INPUT_FILE)

print("Clasificando tipo_falla...")
df["tipo_falla"] = clasificar_batch(df["Summary"].tolist())

print("Procesando vendor/producto...")

results = []

for i in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[i:i+BATCH_SIZE]
    res = procesar_batch(batch["Summary"].tolist())
    results.extend(res)

res_df = pd.DataFrame(results)

df["vendor"] = res_df["v"]
df["producto"] = res_df["p"]

# guardado

df.to_csv(OUTPUT_FILE, index=False)

with open(CACHE_FILE, "w") as f:
    json.dump(cache_data, f)

print("Proceso finalizado")

Hasta acá se tarda todo aprox 70 minutos, la mayoría del tiempo se va en las llamadas a la IA, el embedding y la similitud son relativamente rápidos. El cache ayuda a reducir el tiempo en ejecuciones posteriores.



In [ ]:
# se procede a reprocesar el dataset para completar los faltantes y limpiar un poco los datos.

# mascara para encontrar registros con vendor o producto faltante o invalido

def es_malo(x):
    if pd.isna(x):
        return True
    x = str(x).strip().lower()
    return x in ["", "n/a", "na", "unknown", "error"]

mask_bad = df["vendor"].apply(es_malo) | df["producto"].apply(es_malo)

df_bad = df[mask_bad].copy()
df_good = df[~mask_bad].copy()

print(f"Registros a reprocesar: {len(df_bad)}")

## reprocesamiento de los registros malos

results_bad = []

for i in range(0, len(df_bad), BATCH_SIZE):
    batch = df_bad.iloc[i:i+BATCH_SIZE]
    res = procesar_batch(batch["Summary"].tolist())
    results_bad.extend(res)

res_bad_df = pd.DataFrame(results_bad)

df_bad = df_bad.reset_index(drop=True)
df_bad["vendor"] = res_bad_df["v"]
df_bad["producto"] = res_bad_df["p"]

print("Reprocesamiento finalizado")

# unión de los datos buenos y reprocesados


df_final = pd.concat([df_good, df_bad]).sort_index()

print(f"Dataset final: {len(df_final)} registros")

# limpieza final 

df_final["vendor"] = df_final["vendor"].replace(["", None], "N/A")
df_final["producto"] = df_final["producto"].replace(["", None], "N/A")

df_final["vendor"] = df_final["vendor"].fillna("N/A")
df_final["producto"] = df_final["producto"].fillna("N/A")


In [ ]:
# metricas de calidad

total = len(df_final)

valid_vendor = (~df_final["vendor"].isin(["N/A","error"])).sum()
valid_producto = (~df_final["producto"].isin(["N/A","error"])).sum()

print(f"% Vendor válido: {valid_vendor/total:.2%}")
print(f"% Producto válido: {valid_producto/total:.2%}")

# guardado final!

# =========================================
# EXPORT FINAL
# =========================================

df_final.to_csv("cve_final_F.csv", index=False)

with open(CACHE_FILE, "w") as f:
    json.dump(cache_data, f)

print("Archivo final guardado")